In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

import pickle

from sklearn.metrics import root_mean_squared_error

In [2]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/Users/cszczepanek/projects/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1791979857917, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791979857917, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [3]:
def read_parquet(file_link):
    df = pd.read_parquet(file_link)

    df['lpep_dropoff_datetime'] = pd.to_datetime(df['lpep_dropoff_datetime'])
    df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
    
    df['duration'] = df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
    df['duration'] = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df = df[df.duration >= 1]
    df = df[df.duration <= 60]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [4]:
df_train = read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_eval = read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [5]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_eval['PU_DO'] = df_eval['PULocationID'] + '_' + df_eval['DOLocationID']

In [6]:
categorical = ['PU_DO'] #['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

eval_dicts = df_eval[categorical + numerical].to_dict(orient='records')
X_eval = dv.transform(eval_dicts)

In [7]:
import numpy as np

X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

X_eval.indices = X_eval.indices.astype(np.int32)
X_eval.indptr = X_eval.indptr.astype(np.int32)

In [8]:
target = 'duration'
y_train = df_train[target].values
y_eval = df_eval[target].values

In [9]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_eval)

root_mean_squared_error(y_eval, y_pred)

7.7587152100640475

In [10]:
with open('../models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [11]:
with mlflow.start_run():

    mlflow.set_tag("developer", "Cezary")

    mlflow.log_param("train-data-link", "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-link", "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet")
    
    alpha=0.1
    mlflow.log_param("alpha", alpha)

    lr = Lasso(alpha)
    lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_eval)
    
    rmse = root_mean_squared_error(y_eval, y_pred)
    mlflow.log_metric("RMSE", rmse)

    mlflow.log_artifact(local_path="../models/lin_reg.bin", artifact_path="models_pickle")

In [12]:
lr = Ridge()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_eval)

root_mean_squared_error(y_eval, y_pred)

7.681269743902628

In [13]:
import xgboost as xgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [14]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_eval, label=y_eval)

In [15]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, "validation")],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_eval, y_pred)
        mlflow.log_metric("RMSE", rmse)
    return {"loss" : rmse, "status" : STATUS_OK}

In [16]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:squarederror',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

[0]	validation-rmse:9.60039                                                    
[1]	validation-rmse:8.17131                                                    
[2]	validation-rmse:7.43028                                                    
[3]	validation-rmse:7.05435                                                    
[4]	validation-rmse:6.85937                                                    
[5]	validation-rmse:6.75558                                                    
[6]	validation-rmse:6.69706                                                    
[7]	validation-rmse:6.65924                                                    
[8]	validation-rmse:6.63539                                                    
[9]	validation-rmse:6.61899                                                    
[10]	validation-rmse:6.60912                                                   
[11]	validation-rmse:6.60101                                                   
[12]	validation-rmse:6.59571            

KeyboardInterrupt: 

In [ ]:
mlflow.xgboost.autolog(disable=True)

In [ ]:
with mlflow.start_run():

    best_params = {
        "learning_rate" : 0.2948646839243271,
        "max_depth" : 30,
        "min_child_weight" : 1.9805346649872448,
        "objective" : "reg:squarederror",
        "reg_alpha" : 0.06275960619157453,
        "reg_lambda" : 0.22388496105060088,
        "seed" : 42
    }

    mlflow.log_params(best_params)
    
    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, "validation")],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_eval, y_pred)
    mlflow.log_metric("RMSE", rmse)

    with open("../models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)

    mlflow.log_artifact("../models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

In [17]:
logged_model = 'models:/m-27d6a2620cf343fa8faf5bc5848cf85b'

loaded_model = mlflow.pyfunc.load_model(logged_model)

In [18]:
xgboost_model = mlflow.xgboost.load_model(logged_model)
xgboost_model

In [19]:
y_pred = xgboost_model.predict(valid)
y_pred[:10]

array([14.586527 ,  7.2014832, 14.387785 , 24.29462  ,  9.860293 ,
       17.205248 , 11.048476 ,  8.38429  ,  9.754515 , 20.591978 ],
      dtype=float32)